<a href="https://colab.research.google.com/github/JGradone/Topics-Sentinel/blob/main/Activity_2_Redwing_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Activity 2 - Redwing Engineering Data
**Rutgers University Topics in Marine Science**

*Joe Gradone, Rutgers University*

**Redwing** is a Teledyne Webb Research Slocum glider flying the Sentinel mission, a Rutgers / Teledyne Webb attempt to be the first autonomous underwater vehicle to circumnavigate the globe. In this notebook you'll look at Redwing as a **vehicle**:

* How does a glider fly with no propeller?
* How fast does it go up and down, and does that change over the mission?
* Did it stay healthy (watertight, enough battery)?
* What ocean currents did it have to fight?

The ocean measurements (temperature, salinity, density) are in **Activity 3**.

This builds on **Activity 1**, where you first used `load_glider_data()`. We use exactly the same function here.

### Learning goals
By the end you should be able to:
* Download just the data you need from ERDDAP, including server-side filters
* Explain how a Slocum glider uses its buoyancy engine, pitch and fin to fly
* Compare what the glider was **commanded** to do with what it **measured**
* Use engineering data to judge a glider's health and endurance

### How to use this notebook
**Run the cells in order from the top** (Shift + Enter). Later cells use variables made in earlier ones. If something breaks, use *Runtime → Restart session and run all*.

Cells marked **✏️ Your turn** are for you. Change the values marked `# <- change me` and see what happens.

---
## 1. Setup

Colab already has `numpy`, `pandas`, and `matplotlib`. We install two more:
* [`erddapy`](https://ioos.github.io/erddapy/): builds ERDDAP data requests for us
* [`cmocean`](https://matplotlib.org/cmocean/): colormaps designed for oceanography

In [ ]:
%pip install --quiet erddapy cmocean

In [ ]:
import io
from urllib.parse import quote

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from erddapy import ERDDAP
import matplotlib.dates as mdates
import cmocean.cm as cmo

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

---
## 2. The data: Redwing's delayed-mode engineering dataset

During the mission, Redwing sent only a sample of its data home by satellite (**real-time** data). Here we use the **delayed-mode** data: the complete files stored on the glider, with every measurement it logged, about once a second.

The next cell defines the same functions you used in Activity 1:
* `load_glider_data(dataset_id, variables, start, end, filters)` downloads data into a DataFrame
* `list_glider_variables(dataset_id)` lists every variable in a dataset

In this notebook we use `ENGINEERING_ID` (`redwing-20251011T1511-trajectory-raw-delayed`). [Browse it here.](https://slocum-data.marine.rutgers.edu/erddap/tabledap/redwing-20251011T1511-trajectory-raw-delayed.html)

In [ ]:
SERVER = 'https://slocum-data.marine.rutgers.edu/erddap'
ENGINEERING_ID = 'redwing-20251011T1511-trajectory-raw-delayed'  # everything the glider logged, ~once a second
SCIENCE_ID = 'redwing-20251011T1511-profile-sci-delayed'         # processed CTD profiles with quality flags


def read_csv_from_url(url, **kwargs):
    """pd.read_csv for a web address, with a fallback for servers whose SSL certificate
    Python can't verify. That's fine for public, read-only data like this."""
    try:
        return pd.read_csv(url, **kwargs)
    except Exception as err:
        if 'CERTIFICATE_VERIFY_FAILED' not in str(err):
            raise
        requests.packages.urllib3.disable_warnings()
        response = requests.get(url, verify=False, timeout=600)
        response.raise_for_status()
        return pd.read_csv(io.StringIO(response.text), **kwargs)


def load_glider_data(dataset_id, variables, start=None, end=None, filters=None, server=SERVER):
    """Download glider variables from ERDDAP and return a pandas DataFrame.

    dataset_id : which dataset, e.g. ENGINEERING_ID or SCIENCE_ID
    variables  : list of variable names, e.g. ['time', 'm_depth']
    start, end : optional time limits as strings, e.g. '2025-10-20'
    filters    : optional list of extra ERDDAP filters that run on the server, e.g.
                 ['m_depth!=NaN'] keeps only rows where m_depth has a value
    """
    if 'time' not in variables:
        variables = ['time'] + list(variables)

    e = ERDDAP(server=server, protocol='tabledap', response='csv')
    e.dataset_id = dataset_id
    e.variables = variables
    constraints = {}
    if start is not None:
        constraints['time>='] = start
    if end is not None:
        constraints['time<='] = end
    e.constraints = constraints

    url = e.get_download_url()
    for f in filters or []:
        url += '&' + quote(f, safe='=!,()/')
    df = read_csv_from_url(url, skiprows=[1])  # ERDDAP puts units in the 2nd row; skip it
    df['time'] = pd.to_datetime(df['time']).dt.tz_localize(None)  # all times are UTC
    return df.sort_values('time').reset_index(drop=True)


def list_glider_variables(dataset_id, server=SERVER):
    """Return a table of every variable in a dataset with its units and long name."""
    e = ERDDAP(server=server, protocol='tabledap')
    info = read_csv_from_url(e.get_info_url(dataset_id, response='csv'))
    attrs = info[info['Row Type'] == 'attribute']
    table = attrs.pivot_table(index='Variable Name', columns='Attribute Name',
                              values='Value', aggfunc='first')
    names = info.loc[info['Row Type'] == 'variable', 'Variable Name']
    return table.reindex(names)[['units', 'long_name']]

### What variables are there?

Slocum variable names follow a pattern:

| Prefix | Meaning | Example |
|---|---|---|
| `m_` | **measured** by the glider's flight computer | `m_depth`, `m_pitch` |
| `c_` | **commanded**: what the glider was *told* to do | `c_pitch`, `c_heading` |
| `sci_` | measured by the **science** computer (the sensors) | `sci_water_temp` |

The ones we'll use:

| Variable | What it is | Units |
|---|---|---|
| `m_depth` | glider depth from its own pressure sensor | m |
| `m_de_oil_vol` | oil pumped into the external bladder by the **deep buoyancy engine**. Positive = more buoyant (go up), negative = less buoyant (go down) | cc |
| `m_pitch`, `c_pitch` | nose up (+) or down (−): measured and commanded | radians |
| `m_roll` | tilt side to side | radians |
| `m_heading`, `c_heading` | compass direction the glider points: measured and commanded | radians |
| `m_fin`, `c_fin` | rudder (fin) angle: measured and commanded | radians |
| `m_vacuum` | internal hull vacuum. A steady drop toward 0 can mean a **leak** | inHg |
| `m_water_vx`, `m_water_vy` | depth-averaged current (east, north) over each dive segment | m/s |

Run the next cell to see all of them.

In [ ]:
list_glider_variables(ENGINEERING_ID)

### ⚡ Asking ERDDAP for less data

The glider logs about once a second for months, which adds up to millions of rows. The `filters` argument asks ERDDAP to trim the data **on the server**, before it's sent to us:

| Filter | What it does |
|---|---|
| `'m_depth!=NaN'` | only rows where `m_depth` has a value |
| `'orderByClosest("time/1hour")'` | one row per hour (the one closest to each hour) |

(ERDDAP has lots more. See [the ERDDAP docs](https://slocum-data.marine.rutgers.edu/erddap/tabledap/documentation.html#orderByClosest).)

---
## 3. Mission overview

Where did Redwing go, and how deep did it dive? We'll get the position once an hour and the glider depth every time it was recorded.

In [ ]:
track = load_glider_data(ENGINEERING_ID, ['latitude', 'longitude'],
                         filters=['orderByClosest("time/1hour")'])
depth = load_glider_data(ENGINEERING_ID, ['m_depth'], filters=['m_depth!=NaN'])

print(f'Track: {len(track):,} hourly positions')
print(f'Depth: {len(depth):,} measurements from {depth.time.min()} to {depth.time.max()} (UTC)')

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
sc = ax.scatter(track.longitude, track.latitude, c=mdates.date2num(track.time), s=4, cmap='viridis')
ax.plot(track.longitude.iloc[0], track.latitude.iloc[0], 'g*', ms=16, mec='k', label='start')
ax.plot(track.longitude.iloc[-1], track.latitude.iloc[-1], 'r*', ms=16, mec='k', label='end')
cb = fig.colorbar(sc, ax=ax, label='Date')
cb.ax.yaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
ax.set_aspect(1 / np.cos(np.radians(track.latitude.mean())))  # so degrees of lon & lat look the same length
ax.set_xlabel('Longitude (°E)')
ax.set_ylabel('Latitude (°N)')
ax.set_title('Redwing track')
ax.legend()
plt.show()

In [ ]:
def distance_km(lat, lon):
    """Great-circle distance (km) between consecutive latitude/longitude points."""
    lat, lon = np.radians(lat), np.radians(lon)
    a = (np.sin(np.diff(lat) / 2) ** 2
         + np.cos(lat[:-1]) * np.cos(lat[1:]) * np.sin(np.diff(lon) / 2) ** 2)
    return 2 * 6371 * np.arcsin(np.sqrt(a))


total_km = np.nansum(distance_km(track.latitude.values, track.longitude.values))
days = (track.time.max() - track.time.min()).total_seconds() / 86400
print(f'Mission record: {days:.0f} days, about {total_km:,.0f} km along track ({total_km / days:.0f} km/day)')
print(f'Deepest glider depth: {depth.m_depth.max():.0f} m')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(depth.time, depth.m_depth, '.', ms=0.5)
ax.invert_yaxis()
ax.set_ylabel('Depth (m)')
ax.set_title('Glider depth over the whole mission')
plt.show()

Each thin vertical streak is one **dive and climb**: the glider flies a sawtooth, or "yo", pattern down to ~900 m and back.

🤔 **Discuss:** What changes after mid-January? What might the glider be doing from February to May? (Hint: zoom in with `ax.set_xlim(pd.Timestamp('2026-02-01'), pd.Timestamp('2026-05-20'))` and `ax.set_ylim(20, -1)`.)

---
## 4. How does a glider fly?

A glider has **no propeller**. It moves by changing its buoyancy:
1. At the surface, the **buoyancy engine** pulls oil *into* the hull (`m_de_oil_vol` goes negative). The glider is now denser than seawater and **sinks**.
2. An internal battery pack slides forward to pitch the nose **down** (`m_pitch` negative).
3. The wings turn some of that sinking into forward motion, like a paper airplane.
4. At the bottom, the pump pushes oil *out* into an external bladder (`m_de_oil_vol` positive), the nose pitches **up**, and the glider **climbs**.
5. The whole time, the **fin** (a rudder on the tail) steers to hold the commanded heading.

Let's watch a few dives at full resolution. Angles come in radians, so we convert them to degrees.

In [ ]:
# ✏️ Pick a time window. Try a few different ones (Redwing dove from Oct 2025 to mid-Jan 2026)
start = '2025-10-20 00:00'  # <- change me
end   = '2025-10-20 08:00'  # <- change me

flight = load_glider_data(ENGINEERING_ID,
                          ['m_depth', 'm_de_oil_vol', 'm_pitch', 'c_pitch', 'm_roll',
                           'm_heading', 'c_heading', 'm_fin', 'c_fin'],
                          start=start, end=end)
for angle in ['m_pitch', 'c_pitch', 'm_roll', 'm_heading', 'c_heading', 'm_fin', 'c_fin']:
    flight[angle] = np.degrees(flight[angle])

print(f'{len(flight):,} rows in {start} to {end}')

In [ ]:
def plot_measured_vs_commanded(ax, df, measured, commanded=None, label='', **kwargs):
    """Plot a measured variable (solid) and, optionally, its commanded value (dashed)."""
    d = df.dropna(subset=[measured])
    ax.plot(d.time, d[measured], '-', label=f'measured ({measured})', **kwargs)
    if commanded is not None:
        d = df.dropna(subset=[commanded])
        ax.plot(d.time, d[commanded], '--', color='k', lw=1, label=f'commanded ({commanded})')
        ax.legend(loc='upper right', fontsize=8)
    ax.set_ylabel(label)


fig, axs = plt.subplots(5, 1, figsize=(11, 13), sharex=True)
plot_measured_vs_commanded(axs[0], flight, 'm_depth', label='Depth (m)')
axs[0].invert_yaxis()
plot_measured_vs_commanded(axs[1], flight, 'm_de_oil_vol', label='Oil volume (cc)', color='C1')
axs[1].axhline(0, color='k', lw=0.8)
plot_measured_vs_commanded(axs[2], flight, 'm_pitch', 'c_pitch', label='Pitch (°)', color='C3')
axs[2].axhline(0, color='k', lw=0.8)
plot_measured_vs_commanded(axs[3], flight, 'm_heading', 'c_heading', label='Heading (°)', color='C2')
plot_measured_vs_commanded(axs[4], flight, 'm_fin', 'c_fin', label='Fin (°)', color='C4')

axs[-1].xaxis.set_major_formatter(mdates.DateFormatter('%m-%d\n%H:%M'))
fig.suptitle(f'Redwing flight: {start} to {end} UTC')
fig.tight_layout()
plt.show()

🤔 **Discuss:**
* What happens to the oil volume right before the glider turns around at the bottom? At the top?
* Does the glider reach the pitch it was commanded? Is it the same going down as coming up?
* How closely does the measured heading follow the commanded heading? What is the fin doing when they disagree?

### Vertical speed

We can compute how fast the glider moves up and down from the change in depth over time, using the whole-mission `depth` table from Section 3.

In [ ]:
w = depth.copy()
w['w'] = -w.m_depth.diff() / w.time.diff().dt.total_seconds()  # m/s, positive = up
w = w[(w.m_depth > 50) & w.w.between(-1, 1)]                     # ignore near-surface & glitches

fig, ax = plt.subplots(figsize=(6, 6))
hb = ax.hexbin(w.w, w.m_depth, gridsize=60, bins='log', cmap=cmo.amp, mincnt=1)
fig.colorbar(hb, ax=ax, label='Number of measurements')
ax.invert_yaxis()
ax.axvline(0, color='k', lw=0.8)
ax.set_xlabel('Vertical speed (m/s, + up)')
ax.set_ylabel('Depth (m)')
ax.set_title('Glider vertical speed, whole mission')
plt.show()

print(f"Typical dive speed:  {-w.loc[w.w < 0, 'w'].median():.3f} m/s downward")
print(f"Typical climb speed: {w.loc[w.w > 0, 'w'].median():.3f} m/s upward")

Did the glider slow down as the mission went on? (A glider can get slower if things grow on it: **biofouling**.)

In [ ]:
daily = w.assign(direction=np.where(w.w < 0, 'dive', 'climb'), day=w.time.dt.floor('D'))
daily = daily.groupby(['day', 'direction']).w.median().abs().unstack()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(daily.index, daily.dive, '.-', label='dive')
ax.plot(daily.index, daily.climb, '.-', label='climb')
ax.set_ylabel('Daily median speed (m/s)')
ax.set_title('Vertical speed over the mission')
ax.legend()
plt.show()

---
## 5. Glider health

### Vacuum
The hull is sealed with a partial **vacuum** inside. If seawater leaks in, the vacuum drops toward 0 inHg. Vacuum also changes with the temperature inside the hull (warm air expands), so a slow wiggle is normal; a steady drift is not.

In [ ]:
vacuum = load_glider_data(ENGINEERING_ID, ['m_vacuum'], filters=['m_vacuum!=NaN'])

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(vacuum.time, vacuum.m_vacuum, '.', ms=1, color='C2')
ax.set_ylabel('Vacuum (inHg)')
ax.set_title('Redwing internal vacuum')
plt.show()

### Battery (from the real-time dataset)

The battery decides how long a mission can last. The delayed-mode files on ERDDAP **don't include the battery variables**, but the real-time dataset does. That's no problem: `load_glider_data()` works with any dataset ID.

In [ ]:
REALTIME_ID = 'redwing-20251011T1511-trajectory-raw-rt'  # the data Redwing sent home by satellite

battery = load_glider_data(REALTIME_ID, ['m_battery', 'm_battery_amp_hours_remaining', 'm_veh_temp'])

fig, axs = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
d = battery.dropna(subset=['m_battery_amp_hours_remaining'])
axs[0].plot(d.time, d.m_battery_amp_hours_remaining, '.', ms=2)
axs[0].set_ylabel('Charge remaining\n(amp-hours)')
d = battery.dropna(subset=['m_battery'])
axs[1].plot(d.time, d.m_battery, '.', ms=2, color='C1')
axs[1].set_ylabel('Battery (V)')
d = battery.dropna(subset=['m_veh_temp'])
axs[2].plot(d.time, d.m_veh_temp, '.', ms=2, color='C3')
axs[2].set_ylabel('Hull temp (°C)')
fig.suptitle('Redwing battery and hull temperature (real-time data)')
fig.tight_layout()
plt.show()

---
## 6. Currents: what was the glider fighting?

Each time it surfaces, the glider compares where it *expected* to be (from dead reckoning) with where its GPS says it *is*. The difference is the **depth-averaged current** (`m_water_vx` east, `m_water_vy` north). Slocum gliders only swim at about 0.3–0.4 m/s horizontally, so strong currents matter a lot!

In [ ]:
cur = load_glider_data(ENGINEERING_ID, ['latitude', 'longitude', 'm_water_vx', 'm_water_vy'],
                       filters=['m_water_vx!=NaN'])
cur = cur[(cur.m_water_vx != 0) | (cur.m_water_vy != 0)]
speed = np.hypot(cur.m_water_vx, cur.m_water_vy)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(track.longitude, track.latitude, color='0.7', lw=1, zorder=0)
q = ax.quiver(cur.longitude, cur.latitude, cur.m_water_vx, cur.m_water_vy, speed,
              cmap=cmo.speed, scale=25, width=0.0025)
ax.quiverkey(q, 0.1, 0.92, 0.5, '0.5 m/s', coordinates='axes')
fig.colorbar(q, ax=ax, label='Current speed (m/s)')
ax.set_aspect(1 / np.cos(np.radians(track.latitude.mean())))
ax.set_xlabel('Longitude (°E)')
ax.set_ylabel('Latitude (°N)')
ax.set_title('Depth-averaged currents along the track')
plt.show()

print(f'{len(cur):,} current estimates. Median speed {speed.median():.2f} m/s, max {speed.max():.2f} m/s')

🤔 **Discuss:** Where were the strongest currents? Which major current system is this? (Hint: zoom in with `ax.set_xlim(...)` and `ax.set_ylim(...)`.)

---
## ✏️ Your turn (Engineering group)

1. **Battery budget:** Fit a straight line to amp-hours remaining vs. time (hint: `np.polyfit`) for the diving part of the mission (before mid-January). How many amp-hours per day does Redwing use while diving? Compare with February–May. At each rate, when would the battery run out?
2. **Leak check:** Download `m_vacuum` from `ENGINEERING_ID` and `m_veh_temp` from `REALTIME_ID`, match them up in time (hint: `pd.merge_asof`), and plot vacuum against hull temperature. Is the vacuum change explained by temperature?
3. **Steering:** Using `flight`, compute the heading error `((m_heading - c_heading + 180) % 360) - 180`. When is it biggest? Does it get bigger when the current is strong?
4. **Buoyancy vs. speed:** Does more oil pushed out mean a faster climb? Match `m_de_oil_vol` with vertical speed for a few days and make a scatter plot.
5. **Biofouling:** Did dive or climb speed change between the first and last weeks of diving? By how much?

In [ ]:
# Example to get you started on #1: amp-hours used per day, using all the data
d = battery.dropna(subset=['m_battery_amp_hours_remaining'])
days_since_start = (d.time - d.time.min()).dt.total_seconds() / 86400
slope, intercept = np.polyfit(days_since_start, d.m_battery_amp_hours_remaining, 1)
print(f'Average use: {-slope:.2f} amp-hours per day')

# Now try it for only part of the mission...

---
## 7. Going further
* **More variables:** `list_glider_variables(ENGINEERING_ID)` lists everything. Try `m_iridium_signal_strength`, `m_gps_status`, or `m_dr_x_actual_err` / `m_dr_y_actual_err` (how far off the glider's dead-reckoned position was).
* **Real-time vs. delayed:** Download the same day from `REALTIME_ID` and `ENGINEERING_ID`. How much data did the glider leave out when sending home by satellite?
* **Other gliders:** browse all datasets at <https://slocum-data.marine.rutgers.edu/erddap/tabledap/index.html> and pass a different dataset ID to `load_glider_data(...)`.
* **Science data:** Activity 3 looks at what the CTD measured on the same dives.